# Start

List the layers, open a vector table, explore a tehsil STAC collection and make your first API request.

Run the cells in order. Each step uses data from the previous cells. You can edit the place, identifier, columns and chart settings as you go.


## Set up Python

Run these two collapsed cells once. They load the libraries and starting location. Expand them to see or change the setup.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import ast
import json
from getpass import getpass
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
GEOSERVER = 'https://geoserver.core-stack.org:8443/geoserver/'
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


## Choose the tehsil

The template defaults to Hilsa, Nalanda, Bihar. GeoLibre downloads use your selected place instead. Edit `SCOPE` in the setup cell to change tehsil, then restart the kernel and run from the top. Layer coverage can differ between places.


In [ ]:
state = SCOPE["state"].lower().replace(" ", "_")
district = SCOPE["district"].lower().replace(" ", "_")
tehsil = SCOPE["tehsil"].lower().replace(" ", "_")
place = {"state": state, "district": district, "tehsil": tehsil}
place


## Set your API key for this session

The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains how to register, generate an API key and use it. The key goes in the `X-API-Key` header. This cell stores `CORE_STACK_API_KEY` in the current Python kernel’s environment, so later API cells can reuse it. An existing key is reused without prompting. You can skip this cell when exploring only GeoServer or STAC data. Restarting the kernel may require entering the key again.


In [ ]:
from inspect import isawaitable

api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key

os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()


## List the GeoServer layers

The reference list below contains layer names and services. WFS links return vector data; WMS links show map images. Expand the reference cell to inspect the names.


In [ ]:
layer_reference = [
    ('Administrative Boundaries', 'WFS', 'panchayat_boundaries', '{district}_{tehsil}'),
    ('Socio-Economic Profile', 'WFS', 'panchayat_boundaries', '{district}_{tehsil}'),
    ('Facilities Proximity', 'WFS', 'facilities_proximity', 'facilities_{district}_{tehsil}'),
    ('Mission Antyodaya (2020)', 'WFS', 'antyodaya_2020', 'antyodaya20_{district}_{tehsil}'),
    ('Village Livestock Census', 'WFS', 'livestocks', 'livestocks_{district}_{tehsil}'),
    ('MicroWatershed Boundaries', 'WFS', 'mws_layers', 'deltaG_well_depth_{district}_{tehsil}'),
    ('Annual Water Balance', 'WFS', 'mws_layers', 'deltaG_well_depth_{district}_{tehsil}'),
    ('Fortnightly Water Balance', 'WFS', 'mws_layers', 'deltaG_fortnight_{district}_{tehsil}'),
    ('Terrain Clusters', 'WFS', 'terrain', '{district}_{tehsil}_cluster'),
    ('Drainage Lines', 'WFS', 'drainage', '{district}_{tehsil}'),
    ('Rivers', 'WFS', 'river', '{district}_{tehsil}_river_vector'),
    ('Canals', 'WFS', 'canal', '{district}_{tehsil}_canal_vector'),
    ('Surface Water Bodies', 'WFS', 'swb', 'surface_waterbodies_{district}_{tehsil}'),
    ('Stage of Groundwater Extraction', 'WFS', 'soge', 'soge_vector_{district}_{tehsil}'),
    ('Aquifer', 'WFS', 'aquifer', 'aquifer_vector_{district}_{tehsil}'),
    ('Cropping Intensity', 'WFS', 'crop_intensity', '{district}_{tehsil}_intensity'),
    ('Drought', 'WFS', 'drought', '{district}_{tehsil}_drought'),
    ('Land restoration', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Household livelihood', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Irrigation — site-level impact', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Irrigation — non-RWH', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Community assets', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Plantation and forestry', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Soil and water conservation', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Other or unclassified', 'WFS', 'nrega_assets', '{district}_{tehsil}'),
    ('Green Credit Projects', 'WFS', 'green_credit', '{district}_{tehsil}_green_credit'),
    ('Land Conflicts', 'WFS', 'lcw', '{district}_{tehsil}_lcw_conflict'),
    ('Industries and CSR', 'WFS', 'factory_csr', '{district}_{tehsil}_factory_csr'),
    ('Mining Sites', 'WFS', 'mining', '{district}_{tehsil}_mining'),
    ('Terrain', 'WMS', 'terrain', '{district}_{tehsil}_terrain_raster'),
    ('Digital Elevation Model', 'WMS', 'dem', '{district}_{tehsil}_dem_raster'),
    ('CLART', 'WMS', 'clart', '{district}_{tehsil}_clart'),
    ('Change Detection: Afforestation', 'WMS', 'change_detection', 'change_{district}_{tehsil}_Afforestation'),
    ('Change Detection: Deforestation', 'WMS', 'change_detection', 'change_{district}_{tehsil}_Deforestation'),
    ('Change Detection: Degradation', 'WMS', 'change_detection', 'change_{district}_{tehsil}_Degradation'),
    ('Change Detection: Urbanization', 'WMS', 'change_detection', 'change_{district}_{tehsil}_Urbanization'),
    ('Change Detection: Crop Intensity', 'WMS', 'change_detection', 'change_{district}_{tehsil}_CropIntensity'),
    ('Restoration Opportunities', 'WMS', 'restoration', 'restoration_{district}_{tehsil}_raster'),
    ('LULC Level 1 · 2017-2018', 'WMS', 'LULC_level_3', 'LULC_17_18_{district}_{tehsil}_level_3'),
    ('LULC Level 1 · 2018-2019', 'WMS', 'LULC_level_3', 'LULC_18_19_{district}_{tehsil}_level_3'),
    ('LULC Level 1 · 2019-2020', 'WMS', 'LULC_level_3', 'LULC_19_20_{district}_{tehsil}_level_3'),
    ('LULC Level 1 · 2020-2021', 'WMS', 'LULC_level_3', 'LULC_20_21_{district}_{tehsil}_level_3'),
    ('LULC Level 1 · 2021-2022', 'WMS', 'LULC_level_3', 'LULC_21_22_{district}_{tehsil}_level_3'),
    ('LULC Level 1 · 2022-2023', 'WMS', 'LULC_level_3', 'LULC_22_23_{district}_{tehsil}_level_3'),
    ('LULC Level 1 · 2023-2024', 'WMS', 'LULC_level_3', 'LULC_23_24_{district}_{tehsil}_level_3'),
    ('LULC Level 1 · 2024-2025', 'WMS', 'LULC_level_3', 'LULC_24_25_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2017-2018', 'WMS', 'LULC_level_3', 'LULC_17_18_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2018-2019', 'WMS', 'LULC_level_3', 'LULC_18_19_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2019-2020', 'WMS', 'LULC_level_3', 'LULC_19_20_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2020-2021', 'WMS', 'LULC_level_3', 'LULC_20_21_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2021-2022', 'WMS', 'LULC_level_3', 'LULC_21_22_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2022-2023', 'WMS', 'LULC_level_3', 'LULC_22_23_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2023-2024', 'WMS', 'LULC_level_3', 'LULC_23_24_{district}_{tehsil}_level_3'),
    ('LULC Level 2 · 2024-2025', 'WMS', 'LULC_level_3', 'LULC_24_25_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2017-2018', 'WMS', 'LULC_level_3', 'LULC_17_18_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2018-2019', 'WMS', 'LULC_level_3', 'LULC_18_19_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2019-2020', 'WMS', 'LULC_level_3', 'LULC_19_20_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2020-2021', 'WMS', 'LULC_level_3', 'LULC_20_21_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2021-2022', 'WMS', 'LULC_level_3', 'LULC_21_22_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2022-2023', 'WMS', 'LULC_level_3', 'LULC_22_23_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2023-2024', 'WMS', 'LULC_level_3', 'LULC_23_24_{district}_{tehsil}_level_3'),
    ('LULC Level 3 · 2024-2025', 'WMS', 'LULC_level_3', 'LULC_24_25_{district}_{tehsil}_level_3'),
    ('Micro-watersheds and basins', 'WFS', 'mws', 'mws_{district}_{tehsil}'),
    ('Upstream and downstream micro-watersheds', 'WFS', 'mws_connectivity', '{district}_{tehsil}_mws_connectivity'),
    ('Elevation summary', 'WFS', 'dem', '{district}_{tehsil}_dem_vector'),
    ('Drainage density', 'WFS', 'drainage_density', '{district}_{tehsil}_drainage_density'),
    ('Stream-order shares', 'WFS', 'stream_order', 'stream_order_{district}_{tehsil}_vector'),
    ('Land-cover areas', 'WFS', 'lulc_vector', 'lulc_vector_{district}_{tehsil}'),
    ('NDVI on crop', 'WFS', 'ndvi_timeseries', 'ndvi_timeseries_{district}_{tehsil}_crop'),
    ('NDVI on tree', 'WFS', 'ndvi_timeseries', 'ndvi_timeseries_{district}_{tehsil}_tree'),
    ('NDVI on shrub', 'WFS', 'ndvi_timeseries', 'ndvi_timeseries_{district}_{tehsil}_shrub'),
]


In [ ]:
layer_list = pd.DataFrame(layer_reference, columns=["Layer", "Service", "Workspace", "Layer name"])
layer_list["Layer name"] = layer_list["Layer name"].str.replace("{district}", district, regex=False).str.replace("{tehsil}", tehsil, regex=False)
with pd.option_context("display.max_rows", None):
    display(layer_list)


## See the vector download links

The request names a workspace and layer, asks for features, and chooses GeoJSON as the output format.


In [ ]:
vector_layers = layer_list.loc[layer_list["Service"] == "WFS"].copy()
vector_layers["GeoJSON URL"] = (GEOSERVER + "ows?service=WFS&version=2.0.0&request=GetFeature&typeNames="
    + vector_layers["Workspace"] + ":" + vector_layers["Layer name"]
    + "&outputFormat=application/json&srsName=EPSG:4326")
with pd.option_context("display.max_rows", None):
    display(vector_layers[["Layer", "GeoJSON URL"]])


## Open the micro-watershed layer

Read one specific link. The JSON response contains `features`, which GeoPandas turns into rows and a geometry column.


In [ ]:
mws_url = f"{GEOSERVER}mws/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=mws:mws_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326"
response = requests.get(mws_url, timeout=90)
response.raise_for_status()
mws_geojson = response.json()
mws = gpd.GeoDataFrame.from_features(mws_geojson["features"], crs="EPSG:4326")
mws.head()


## Read a few columns

`uid` identifies a micro-watershed. `area_in_ha` gives its area in hectares. Change the column list to inspect other fields.


In [ ]:
mws.drop(columns="geometry").iloc[:10, :10]


## Save the data

GeoJSON keeps the shapes; CSV keeps the table. Download the saved files from the notebook file browser. Open the GeoJSON using [GeoLibre’s Add Data tools](https://geolibre.app/user-guide/interface/) or follow the [CoRE Stack QGIS guide](https://docs.google.com/document/d/1jet4EEBbbKgpNrPnuNJJDRuAJUiR2pIMFQp9JTlygAQ/edit).


In [ ]:
# GeoPandas writes GeoJSON directly, without a separate GIS file driver.
with open("micro-watersheds.geojson", "w") as file:
    file.write(mws.to_json())
mws.drop(columns="geometry").to_csv("micro-watersheds.csv", index=False)
display(FileLink("micro-watersheds.geojson"), FileLink("micro-watersheds.csv"))


## Open the tehsil STAC collection

A STAC collection describes a group of datasets. This URL follows the state/district/tehsil folders in the [CoRE Stack catalogue](https://stac.core-stack.org/tehsil_wise/catalog.json).


In [ ]:
collection_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/collection.json")
collection = requests.get(collection_url, timeout=90).json()
print(collection_url)
display(pd.Series(collection)[["id", "description", "extent", "license"]])


## List the items in this collection

Each `item` link describes a dataset. Relative links are resolved against the collection URL.


In [ ]:
collection_links = pd.DataFrame(collection["links"])
items = collection_links.loc[collection_links["rel"] == "item", ["href", "type"]].copy()
items["Item URL"] = [urljoin(collection_url, href) for href in items["href"]]
with pd.option_context("display.max_rows", None):
    display(items[["Item URL", "type"]])


## Read one item’s metadata

Here we inspect the change-in-cropping-intensity item. Its metadata describes a raster dataset; this step reads only the metadata, not the raster. Choose another item from the preceding list to explore its description.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_change_cropping_intensity_raster"
item_url = urljoin(collection_url, f"{item_name}/{item_name}.json")
item = requests.get(item_url, timeout=90).json()
print(item_url)
display(pd.Series(item["properties"], name="Value").to_frame())


## Find the item’s assets

An item’s `assets` dictionary gives links to its data. Read the media type and description before downloading an asset.


In [ ]:
assets = pd.DataFrame.from_dict(item["assets"], orient="index")
assets["href"] = [urljoin(item_url, href) for href in assets["href"]]
assets


## Explore the public APIs

The [API specifications](https://api-doc.core-stack.org) describe each endpoint. Follow the [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) to generate your API key. This public OpenAPI document lists the callable paths, required parameters and response descriptions; reading it does not require your key.


In [ ]:
API_SCHEMA_URL = "https://geoserver.core-stack.org/?format=openapi"
schema_response = requests.get(API_SCHEMA_URL, timeout=90)
schema_response.raise_for_status()
api_spec = schema_response.json()
api_paths = {path: methods["get"] for path, methods in api_spec["paths"].items() if path.startswith("/get_")}
api_catalogue = pd.DataFrame([
    {"API path": path,
     "Required parameters": ", ".join(p["name"] for p in operation["parameters"] if p["in"] == "query" and p.get("required")),
     "Optional parameters": ", ".join(p["name"] for p in operation["parameters"] if p["in"] == "query" and not p.get("required"))}
    for path, operation in api_paths.items()
])
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(api_catalogue)


## Choose an API and inspect its specification

Change `api_path` to any path in the table. The parameter table shows types and descriptions. The response definitions include documented schemas or examples.


In [ ]:
api_path = "/get_active_locations/"
operation = api_paths[api_path]
print(operation.get("description", operation.get("summary", "")))
parameters = pd.DataFrame(operation["parameters"])
display(parameters.reindex(columns=["name", "in", "required", "type", "description"]))
print(json.dumps(operation["responses"], indent=2))


## Connect to the CoRE Stack API

Read endpoint specifications at [api-doc.core-stack.org](https://api-doc.core-stack.org). The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains how to register, generate an API key and use it. The key goes in the `X-API-Key` header. This cell reads `CORE_STACK_API_KEY` from your environment, or asks for it without showing it. The key is not written into the notebook. After each API request, run the collapsed parsing cell: it keeps the raw response text and reads it with `json.loads`, which accepts `NaN` as a missing numeric value. It also shows HTTP errors and skips dependent API cells if the request fails. Expand the cell to inspect the code.


In [ ]:
from inspect import isawaitable

api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key

api_headers = {"X-API-Key": str(api_key).strip()}


## Set parameters for the chosen API

The same base URL and API key header work for all the listed APIs. Edit `request_params` to match the chosen specification: use `{}` for active locations, `place.copy()` for tehsil APIs, or `{**place, "mws_id": mws_id}` for one MWS. Coordinate APIs take `latitude` and `longitude`; the single-waterbody API takes `place` and a `uid` from Notebook 4.


In [ ]:
mws_id = mws["uid"].sort_values().iloc[0]
request_params = {}  # get_active_locations needs no query parameters.
request_url = API_URL.rstrip("/") + api_path
print(request_url)
request_params


## Request the data

Run this cell after changing the API path and parameters. The JSON response stays available as `api_result` for your next cell. No API key is included in the displayed URL.


In [ ]:
api_response = requests.get(request_url, params=request_params, headers=api_headers, timeout=180)
# The next cell checks the HTTP status and reads the response.


In [ ]:
raw_api_data_string = api_response.text
api_payload = None
if api_response.ok:
    try:
        api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))
    except ValueError:
        print("The API returned a response that is not valid JSON. Preview:", raw_api_data_string[:500])
else:
    print(f"API returned HTTP {api_response.status_code} for {api_response.url}")
    print("This request did not return data. Other API examples can still be run.")
    print(raw_api_data_string[:500])


In [ ]:
if api_payload is not None:
    api_result = api_payload
    display(print(json.dumps(api_result, indent=2)[:4000]))  # Preview; api_result contains the full response.


## Find the generated layer links

`get_generated_layer_urls` lists published layer links for this tehsil, including styling information where supplied.


In [ ]:
response = requests.get(API_URL + "get_generated_layer_urls/", params=place, headers=api_headers, timeout=90)
# The next cell checks the HTTP status and reads the response.


In [ ]:
raw_api_data_string = response.text
api_payload = None
if response.ok:
    try:
        api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))
    except ValueError:
        print("The API returned a response that is not valid JSON. Preview:", raw_api_data_string[:500])
else:
    print(f"API returned HTTP {response.status_code} for {response.url}")
    print("This request did not return data. Other API examples can still be run.")
    print(raw_api_data_string[:500])


In [ ]:
if api_payload is not None:
    api_layers = pd.DataFrame(api_payload)
    display(api_layers.head(10))


## Read the tehsil tables from the API

`get_tehsil_data` returns a dictionary of tables for the same place. Here we make the request once and reuse the returned tables below.


In [ ]:
api_response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
# The next cell checks the HTTP status and reads the response.


In [ ]:
raw_api_data_string = api_response.text
api_payload = None
if api_response.ok:
    try:
        api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))
    except ValueError:
        print("The API returned a response that is not valid JSON. Preview:", raw_api_data_string[:500])
else:
    print(f"API returned HTTP {api_response.status_code} for {api_response.url}")
    print("This request did not return data. Other API examples can still be run.")
    print(raw_api_data_string[:500])


In [ ]:
if api_payload is not None:
    api_data = api_payload
    display(pd.DataFrame({"Table": api_data.keys(), "Rows": [len(rows) for rows in api_data.values()]}))


## Choose a tehsil table

Change `table_name` to any table in the preceding list. Start with the MWS records.


In [ ]:
if api_payload is not None:
    table_name = "mws"
    api_table = pd.DataFrame(api_data[table_name])
    display(api_table.head(10))


## Find an MWS from a point

Use a point inside the first MWS, or change the latitude and longitude to your own location.


In [ ]:
point = mws.geometry.iloc[0].representative_point()
coordinates = {"latitude": point.y, "longitude": point.x}
response = requests.get(API_URL + "get_mwsid_by_latlon/", params=coordinates, headers=api_headers, timeout=90)
# The next cell checks the HTTP status and reads the response.


In [ ]:
raw_api_data_string = response.text
api_payload = None
if response.ok:
    try:
        api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))
    except ValueError:
        print("The API returned a response that is not valid JSON. Preview:", raw_api_data_string[:500])
else:
    print(f"API returned HTTP {response.status_code} for {response.url}")
    print("This request did not return data. Other API examples can still be run.")
    print(raw_api_data_string[:500])


In [ ]:
if api_payload is not None:
    display(api_payload)
